# 01-05 参数初始化

参数初始化研究的是：神经网络训练开始前，权重 $\mathbf{W}$ 和偏置 $\mathbf{b}$ 应该怎么设置。

初始化看起来只是训练前的一小步，但它会直接影响前向传播的信号大小、反向传播的梯度大小，以及模型能不能顺利开始学习。

## 1. 学习目标

学完这一节，需要能回答下面几个问题：

1. 为什么权重不能全部初始化为 $0$？
2. 为什么权重太大或太小都会影响训练？
3. 什么是 fan_in 和 fan_out？
4. Xavier 初始化适合什么激活函数？
5. He 初始化为什么适合 ReLU？
6. 在 PyTorch 中如何手动初始化参数？

## 2. 参数初始化解决什么问题

一层神经网络通常写成：

$$
\mathbf{h}=\phi(\mathbf{W}\mathbf{x}+\mathbf{b})
$$

其中 $\mathbf{W}$ 和 $\mathbf{b}$ 是需要训练的参数，$\phi$ 是激活函数。

训练开始之前，模型还没有见过数据，所以必须先给参数一个初始值。好的初始化希望做到三件事：

1. 打破神经元之间的对称性。
2. 让前向传播时每层输出的尺度不要爆炸或消失。
3. 让反向传播时梯度的尺度也尽量稳定。

如果初始化不好，网络可能一开始就陷入梯度消失、梯度爆炸，或者多个神经元学到完全相同的东西。

## 3. 为什么不能全部初始化为 0

假设隐藏层有两个神经元，它们的初始权重完全一样：

$$
\mathbf{w}_1=\mathbf{w}_2
$$

偏置也完全一样：

$$
b_1=b_2
$$

那么对同一个输入 $\mathbf{x}$，两个神经元的线性输出相同：

$$
z_1=\mathbf{w}_1^T\mathbf{x}+b_1
$$

$$
z_2=\mathbf{w}_2^T\mathbf{x}+b_2
$$

因此：

$$
z_1=z_2
$$

经过同一个激活函数后：

$$
h_1=\phi(z_1), \quad h_2=\phi(z_2)
$$

所以：

$$
h_1=h_2
$$

反向传播时，这两个神经元收到的梯度也会一样，参数更新后仍然一样。它们就像复制出来的同一个神经元，无法学到不同特征。

这叫对称性问题。随机初始化的第一个作用，就是打破这种对称性。

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)

x = torch.tensor([[1.0, 2.0, 3.0]])
layer = nn.Linear(3, 2)

# 人为让两个神经元的权重和偏置完全一样。
with torch.no_grad():
    layer.weight[:] = torch.tensor([[0.1, 0.2, 0.3], [0.1, 0.2, 0.3]])
    layer.bias[:] = torch.tensor([0.5, 0.5])

output = layer(x)
print(output)
print('两个神经元输出是否相等:', torch.allclose(output[:, 0], output[:, 1]))


## 4. 权重太大或太小的问题

考虑一层线性变换：

$$
z=\sum_{i=1}^{n}w_i x_i
$$

如果权重 $w_i$ 太大，$z$ 的数值会越来越大。进入 Sigmoid 或 Tanh 后，容易落到饱和区：

$$
\sigma'(z)\approx 0
$$

$$
\tanh'(z)\approx 0
$$

这会导致梯度消失。

如果权重 $w_i$ 太小，前向传播时每层输出会越来越接近 $0$，信号变弱；反向传播时梯度也可能越来越小。

所以初始化的核心不是“随机就行”，而是要让每层输入输出的方差尽量稳定。

## 5. 从方差角度理解初始化

假设输入 $x_i$ 和权重 $w_i$ 相互独立，且均值都接近 $0$：

$$
\mathbb{E}[x_i]=0, \quad \mathbb{E}[w_i]=0
$$

线性输出为：

$$
z=\sum_{i=1}^{n}w_i x_i
$$

可以近似得到：

$$
\operatorname{Var}(z)=n\operatorname{Var}(w)\operatorname{Var}(x)
$$

其中 $n$ 是输入特征数量，也叫 fan_in：

$$
n=\operatorname{fan\_in}
$$

如果希望输出方差和输入方差差不多：

$$
\operatorname{Var}(z)\approx \operatorname{Var}(x)
$$

那么需要：

$$
n\operatorname{Var}(w)\approx 1
$$

所以：

$$
\operatorname{Var}(w)\approx \frac{1}{n}
$$

这就是很多初始化方法背后的核心思想：根据层的输入输出规模，控制权重方差。

## 6. fan_in 和 fan_out

对于全连接层：

$$
\mathbf{y}=\mathbf{x}\mathbf{W}+\mathbf{b}
$$

如果输入特征数是 $n_{in}$，输出神经元数是 $n_{out}$，那么：

$$
\operatorname{fan\_in}=n_{in}
$$

$$
\operatorname{fan\_out}=n_{out}
$$

对于卷积层，假设卷积核大小是 $k_h\times k_w$，输入通道数是 $c_{in}$，输出通道数是 $c_{out}$，那么：

$$
\operatorname{fan\_in}=c_{in}k_hk_w
$$

$$
\operatorname{fan\_out}=c_{out}k_hk_w
$$

fan_in 主要影响前向传播的输出尺度，fan_out 主要影响反向传播的梯度尺度。

## 7. Xavier 初始化

Xavier 初始化也叫 Glorot 初始化，常用于 Sigmoid 和 Tanh 这类近似对称、容易饱和的激活函数。

它希望同时兼顾前向传播和反向传播，因此同时考虑 fan_in 和 fan_out：

$$
\operatorname{Var}(w)=\frac{2}{\operatorname{fan\_in}+\operatorname{fan\_out}}
$$

如果使用正态分布初始化：

$$
w\sim \mathcal{N}\left(0,\frac{2}{\operatorname{fan\_in}+\operatorname{fan\_out}}\right)
$$

如果使用均匀分布初始化：

$$
w\sim U(-a,a)
$$

其中：

$$
a=\sqrt{\frac{6}{\operatorname{fan\_in}+\operatorname{fan\_out}}}
$$

记忆方式：Tanh / Sigmoid 可以优先想到 Xavier。

## 8. He 初始化

He 初始化也叫 Kaiming 初始化，常用于 ReLU 及其变体。

ReLU 会把负半轴输出变成 $0$，如果输入大致正负对称，约有一半信号被截断。为了补偿这个影响，He 初始化通常让权重方差更大一些：

$$
\operatorname{Var}(w)=\frac{2}{\operatorname{fan\_in}}
$$

如果使用正态分布初始化：

$$
w\sim \mathcal{N}\left(0,\frac{2}{\operatorname{fan\_in}}\right)
$$

对应标准差是：

$$
\operatorname{std}(w)=\sqrt{\frac{2}{\operatorname{fan\_in}}}
$$

记忆方式：ReLU / Leaky ReLU / PReLU / RReLU 可以优先想到 He 初始化。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

def relu(x):
    return np.maximum(0, x)

def tanh(x):
    return np.tanh(x)

def simulate_variance(init_name, activation, width=256, depth=30, batch_size=512):
    x = np.random.randn(batch_size, width)
    variances = []

    for _ in range(depth):
        fan_in = width
        fan_out = width

        if init_name == 'small_random':
            W = np.random.randn(fan_in, fan_out) * 0.01
        elif init_name == 'large_random':
            W = np.random.randn(fan_in, fan_out) * 1.0
        elif init_name == 'xavier':
            std = np.sqrt(2 / (fan_in + fan_out))
            W = np.random.randn(fan_in, fan_out) * std
        elif init_name == 'he':
            std = np.sqrt(2 / fan_in)
            W = np.random.randn(fan_in, fan_out) * std
        else:
            raise ValueError(init_name)

        x = activation(x @ W)
        variances.append(np.var(x))

    return variances

depth = 30
layers = np.arange(1, depth + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for name in ['small_random', 'large_random', 'xavier']:
    axes[0].plot(layers, simulate_variance(name, tanh, depth=depth), label=name)
axes[0].set_title('Tanh: Activation Variance Across Layers')
axes[0].set_xlabel('layer')
axes[0].set_ylabel('variance')
axes[0].set_yscale('log')
axes[0].legend()
axes[0].grid(alpha=0.3)

for name in ['small_random', 'large_random', 'he']:
    axes[1].plot(layers, simulate_variance(name, relu, depth=depth), label=name)
axes[1].set_title('ReLU: Activation Variance Across Layers')
axes[1].set_xlabel('layer')
axes[1].set_ylabel('variance')
axes[1].set_yscale('log')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 9. 偏置如何初始化

偏置 $\mathbf{b}$ 通常比权重简单。常见做法是初始化为 $0$：

$$
\mathbf{b}=\mathbf{0}
$$

这通常不会产生和权重全零初始化一样严重的对称性问题，因为权重 $\mathbf{W}$ 已经是随机的，不同神经元仍然会有不同输出。

在某些特殊场景中，也会对偏置做非零初始化。例如希望 ReLU 神经元一开始更容易激活，可以把偏置设成一个小正数：

$$
b=0.01
$$

但入门阶段先记住：权重认真初始化，偏置通常初始化为 $0$。

## 10. PyTorch 中的参数初始化

PyTorch 在创建 `nn.Linear`、`nn.Conv2d` 等层时，会自动使用默认初始化。学习阶段我们也要知道如何手动初始化。

In [ ]:
import torch
from torch import nn

class SmallMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(20, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 3)
        )

    def forward(self, x):
        return self.net(x)

model = SmallMLP()

def init_weights(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_normal_(module.weight, nonlinearity='relu')
        nn.init.zeros_(module.bias)

model.apply(init_weights)

for name, param in model.named_parameters():
    print(name, tuple(param.shape), 'mean=', round(param.data.mean().item(), 4), 'std=', round(param.data.std().item(), 4))


常见初始化函数：

```python
nn.init.xavier_uniform_(tensor)
nn.init.xavier_normal_(tensor)
nn.init.kaiming_uniform_(tensor, nonlinearity='relu')
nn.init.kaiming_normal_(tensor, nonlinearity='relu')
nn.init.zeros_(tensor)
nn.init.ones_(tensor)
```

如果隐藏层使用 Tanh，可以考虑：

```python
nn.init.xavier_normal_(module.weight)
```

如果隐藏层使用 ReLU，可以考虑：

```python
nn.init.kaiming_normal_(module.weight, nonlinearity='relu')
```

## 11. 实用选择规则

| 激活函数 | 推荐初始化 | 记忆方式 |
|---|---|---|
| Sigmoid | Xavier | S 形函数，控制方差，避免过早饱和 |
| Tanh | Xavier | 零中心 S 形函数，适合 Xavier |
| ReLU | He / Kaiming | 负半轴被截断，需要更大方差补偿 |
| Leaky ReLU | He / Kaiming | 属于 ReLU 变体 |
| PReLU | He / Kaiming | 属于 ReLU 变体 |
| RReLU | He / Kaiming | 属于 ReLU 变体 |

入门阶段可以先记住：

1. 不要把权重全部初始化为 $0$。
2. Tanh / Sigmoid 优先考虑 Xavier。
3. ReLU 及其变体优先考虑 He。
4. 偏置通常可以初始化为 $0$。
5. 如果训练一开始 loss 就异常爆炸或完全不动，要检查初始化、学习率和输入数据尺度。

## 12. 本节总结

参数初始化不是随便给一组随机数，而是在训练开始前尽量保证信号和梯度稳定。

这一节重点记住三个公式：

Xavier 初始化：

$$
\operatorname{Var}(w)=\frac{2}{\operatorname{fan\_in}+\operatorname{fan\_out}}
$$

He 初始化：

$$
\operatorname{Var}(w)=\frac{2}{\operatorname{fan\_in}}
$$

线性输出方差近似：

$$
\operatorname{Var}(z)=n\operatorname{Var}(w)\operatorname{Var}(x)
$$

下一步再进入感知机和多层感知机时，我们就能更清楚地理解：模型不是只要结构写对就能训练好，初始参数也会影响学习能不能顺利开始。